In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1 · Parameters

In [2]:
# Market-impact parameters
A1, A2, A3, A4, B1 = 883.58, 0.35, 0.75, 0.82, 0.96

# Optimization inputs
POV_FIXED = 0.10      # Fixed POV for the base pre-trade metrics
LAMBDA    = 0.25      # Risk-aversion for Trader's Dilemma
BID_BP    = 100.0     # Principal bid (bp) for Price Improvement

# POV search bounds (POV must be in (0, 1); 50% is the standard practical cap)
POV_BOUNDS = (0.01, 0.50)

## 2 · Load the trade list

In [3]:
df = pd.read_csv('Case_Study_02_PreTrade.csv')

# Encode side (+1 for Buy, -1 for Sell) and order size as %ADV (decimal)
df['Side'] = np.where(df['OrderSide'].str.lower() == 'buy', 1, -1)
df['Size'] = df['Shares'] / df['ADV']

df.head()

,Number,TradeDate,Symbol,OrderSide,Shares,Price,Volatility,ADV,AlphaBp,Side,Size
0,1,4/21/2025,IBM,Buy,309300,236.2200,0.3900,5153500,38,1,0.0600
1,2,4/21/2025,TRV,Buy,145730,249.5900,0.3600,1616000,33,1,0.0902
2,3,4/21/2025,EMR,Buy,324790,96.4200,0.5500,3611800,56,1,0.0899
3,4,4/21/2025,IFF,Buy,21280,72.7100,0.4300,2113100,39,1,0.0101
4,5,4/21/2025,SO,Buy,50150,90.2300,0.2300,5008400,22,1,0.0100


## 3 · Core pre-trade formulas

In [4]:
def market_impact(size, sigma, pov):
    """Market Impact (bp).  size, sigma, pov are decimals."""
    I_star = A1 * (size ** A2) * (sigma ** A3)
    return I_star * (B1 * (pov ** A4) + (1.0 - B1))

def timing_risk(size, sigma, pov):
    """Timing Risk (bp)."""
    return sigma * np.sqrt((1.0/3.0) * (1.0/250.0) * size * (1.0 - pov) / pov) * 1e4

def price_appreciation(side, alpha_bp, size, pov):
    """Price Appreciation (bp)."""
    return side * 0.5 * alpha_bp * size * (1.0 - pov) / pov

def trade_time(size, pov):
    """Trade time as a fraction of one trading day."""
    return size * (1.0 - pov) / pov

## 4 · Pre-trade metrics at POV = 10%

In [5]:
df['MI_bp_POV10']     = market_impact(df['Size'], df['Volatility'], POV_FIXED)
df['TR_bp_POV10']     = timing_risk(df['Size'], df['Volatility'], POV_FIXED)
df['PA_bp_POV10']     = price_appreciation(df['Side'], df['AlphaBp'], df['Size'], POV_FIXED)
df['TradeTime_POV10'] = trade_time(df['Size'], POV_FIXED)
df['Size_pctADV']     = df['Size'] * 100.0

df[['Symbol','OrderSide','Size_pctADV','MI_bp_POV10','TR_bp_POV10',
    'PA_bp_POV10','TradeTime_POV10']]

,Symbol,OrderSide,Size_pctADV,MI_bp_POV10,TR_bp_POV10,PA_bp_POV10,TradeTime_POV10
0,IBM,Buy,6.0017,30.1871,104.6632,10.2630,0.5402
1,TRV,Buy,9.0179,32.7824,118.4260,13.3916,0.8116
2,EMR,Buy,8.9925,45.0044,180.6728,22.6610,0.8093
3,IFF,Buy,1.0071,17.3898,47.2699,1.7674,0.0906
4,SO,Buy,1.0013,10.8548,25.2118,0.9913,0.0901
5,MSFT,Buy,1.0018,16.4414,43.8566,2.0286,0.0902
6,DG,Buy,3.9998,29.6420,100.7782,1.4399,0.3600
7,GDDY,Buy,2.9915,24.1118,75.7865,7.5385,0.2692
8,FAST,Buy,2.0018,21.3408,63.5456,3.6933,0.1802
9,BX,Buy,5.9989,48.7948,198.5442,4.8591,0.5399


## 5 · Optimization objectives


In [6]:
def traders_dilemma_obj(pov, size, sigma, lam):
    return market_impact(size, sigma, pov) + lam * timing_risk(size, sigma, pov)

def min_cost_obj(pov, side, size, sigma, alpha_bp):
    return market_impact(size, sigma, pov) + price_appreciation(side, alpha_bp, size, pov)

def price_improvement_obj(pov, size, sigma, bid_bp):
    mi = market_impact(size, sigma, pov)
    tr = timing_risk(size, sigma, pov)
    return (mi - bid_bp) / tr   # minimizing this maximizes (Bid - MI)/TR

def optimize_pov(objective, args, bounds=POV_BOUNDS):
    res = minimize_scalar(objective, args=args, bounds=bounds,
                          method='bounded', options={'xatol': 1e-6})
    return res.x

## 6 · Compute optimal POV for each stock and each strategy

In [7]:
pov_td, pov_mc, pov_pi = [], [], []

for _, r in df.iterrows():
    size, sigma, side, alpha = r['Size'], r['Volatility'], r['Side'], r['AlphaBp']
    pov_td.append(optimize_pov(traders_dilemma_obj, (size, sigma, LAMBDA)))
    pov_mc.append(optimize_pov(min_cost_obj,        (side, size, sigma, alpha)))
    pov_pi.append(optimize_pov(price_improvement_obj, (size, sigma, BID_BP)))

df['POV_TradersDilemma']    = pov_td
df['POV_MinCost']           = pov_mc
df['POV_PriceImprovement']  = pov_pi

df[['Symbol','OrderSide','POV_TradersDilemma','POV_MinCost','POV_PriceImprovement']]

,Symbol,OrderSide,POV_TradersDilemma,POV_MinCost,POV_PriceImprovement
0,IBM,Buy,0.0797,0.0747,0.1906
1,TRV,Buy,0.0823,0.0826,0.1678
2,EMR,Buy,0.0894,0.0926,0.1044
3,IFF,Buy,0.0659,0.0385,0.5000
4,SO,Buy,0.0583,0.0363,0.5000
5,MSFT,Buy,0.0649,0.0428,0.5000
6,DG,Buy,0.0784,0.0256,0.1961
7,GDDY,Buy,0.0738,0.0713,0.2747
8,FAST,Buy,0.0707,0.0515,0.3422
9,BX,Buy,0.0903,0.0380,0.0927


## 7 · Diagnostics — cost & risk at each optimal POV

For each strategy we report MI, TR, PA, and the objective value at the chosen POV.

In [10]:
def metrics_at_pov(row, pov):
    s, v, sd, a = row['Size'], row['Volatility'], row['Side'], row['AlphaBp']
    return pd.Series({
        'MI_bp':     market_impact(s, v, pov),
        'TR_bp':     timing_risk(s, v, pov),
        'PA_bp':     price_appreciation(sd, a, s, pov),
        'TradeTime': trade_time(s, pov),
    })

# Trader's Dilemma diagnostics
td_metrics = df.apply(lambda r: metrics_at_pov(r, r['POV_TradersDilemma']), axis=1)
td_metrics.columns = [c + '_TD' for c in td_metrics.columns]
td_metrics['Obj_TD_bp'] = td_metrics['MI_bp_TD'] + LAMBDA * td_metrics['TR_bp_TD']

# Min-Cost diagnostics
mc_metrics = df.apply(lambda r: metrics_at_pov(r, r['POV_MinCost']), axis=1)
mc_metrics.columns = [c + '_MC' for c in mc_metrics.columns]
mc_metrics['Obj_MC_bp'] = mc_metrics['MI_bp_MC'] + mc_metrics['PA_bp_MC']

# Price Improvement diagnostics
pi_metrics = df.apply(lambda r: metrics_at_pov(r, r['POV_PriceImprovement']), axis=1)
pi_metrics.columns = [c + '_PI' for c in pi_metrics.columns]
pi_metrics['Obj_PI'] = (BID_BP - pi_metrics['MI_bp_PI']) / pi_metrics['TR_bp_PI']

diagnostics = pd.concat(
    [df[['Symbol','OrderSide','POV_TradersDilemma','POV_MinCost','POV_PriceImprovement']],
     td_metrics, mc_metrics, pi_metrics], axis=1
)
diagnostics.head(10)

,Symbol,OrderSide,POV_TradersDilemma,POV_MinCost,POV_PriceImprovement,MI_bp_TD,TR_bp_TD,PA_bp_TD,TradeTime_TD,Obj_TD_bp,MI_bp_MC,TR_bp_MC,PA_bp_MC,TradeTime_MC,Obj_MC_bp,MI_bp_PI,TR_bp_PI,PA_bp_PI,TradeTime_PI,Obj_PI
0,IBM,Buy,0.0797,0.0747,0.1906,26.1587,118.5906,13.1761,0.6935,55.8063,25.1431,122.8244,14.1337,0.7439,39.2767,46.6817,71.9018,4.8436,0.2549,0.7415
1,TRV,Buy,0.0823,0.0826,0.1678,28.9781,131.8558,16.6012,1.0061,61.9420,29.0493,131.5716,16.5297,1.0018,45.5790,46.3704,87.9165,7.3804,0.4473,0.6100
2,EMR,Buy,0.0894,0.0926,0.1044,41.8966,192.2456,25.6570,0.9163,89.9580,42.8602,188.4784,24.6614,0.8808,67.5215,46.2867,176.3459,21.5886,0.7710,0.3046
3,IFF,Buy,0.0659,0.0385,0.5000,13.4369,59.3366,2.7849,0.1428,28.2710,9.9816,78.7917,4.9104,0.2518,14.8921,54.7857,15.7567,0.1964,0.0101,2.8695
4,SO,Buy,0.0583,0.0363,0.5000,7.8110,33.7781,1.7794,0.1618,16.2555,6.0477,43.3267,2.9276,0.2661,8.9753,34.1975,8.4040,0.1101,0.0100,7.8300
5,MSFT,Buy,0.0649,0.0428,0.5000,12.5945,55.4861,3.2471,0.1443,26.4660,9.9749,69.1534,5.0437,0.2242,15.0186,51.7979,14.6189,0.2254,0.0100,3.2973
6,DG,Buy,0.0784,0.0256,0.1961,25.4451,115.1442,1.8797,0.4699,54.2311,14.0106,207.1170,6.0819,1.5205,20.0925,46.7713,68.0205,0.6560,0.1640,0.7825
7,GDDY,Buy,0.0738,0.0713,0.2747,19.9373,89.5151,10.5170,0.3756,42.3160,19.5316,91.1736,10.9103,0.3897,30.4419,48.4990,41.0532,2.2120,0.0790,1.2545
8,FAST,Buy,0.0707,0.0515,0.3422,17.2023,76.7841,5.3925,0.2630,36.3984,14.3209,90.8869,7.5553,0.3685,21.8762,50.4952,29.3683,0.7889,0.0385,1.6857
9,BX,Buy,0.0903,0.0380,0.0927,45.7297,210.0358,5.4378,0.6042,98.2386,27.8481,332.8753,13.6585,1.5176,41.5067,46.4932,207.0321,5.2834,0.5870,0.2584


## 8 · Final results table

In [11]:
results = df[[
    'Number','TradeDate','Symbol','OrderSide','Shares','Price',
    'Volatility','ADV','AlphaBp','Size_pctADV',
    'MI_bp_POV10','TR_bp_POV10','PA_bp_POV10','TradeTime_POV10',
    'POV_TradersDilemma','POV_MinCost','POV_PriceImprovement',
]].copy()

results = results.rename(columns={
    'Size_pctADV':           'Size_%ADV',
    'MI_bp_POV10':           'MI_bp_(POV=10%)',
    'TR_bp_POV10':           'TR_bp_(POV=10%)',
    'PA_bp_POV10':           'PA_bp_(POV=10%)',
    'TradeTime_POV10':       'TradeTime_(POV=10%)',
    'POV_TradersDilemma':    'OptPOV_TradersDilemma_(lambda=0.25)',
    'POV_MinCost':           'OptPOV_MinCost_(AlphaBp)',
    'POV_PriceImprovement':  'OptPOV_PriceImp_(Bid=100bp)',
})

results

,Number,TradeDate,Symbol,OrderSide,Shares,Price,Volatility,ADV,AlphaBp,Size_%ADV,MI_bp_(POV=10%),TR_bp_(POV=10%),PA_bp_(POV=10%),TradeTime_(POV=10%),OptPOV_TradersDilemma_(lambda=0.25),OptPOV_MinCost_(AlphaBp),OptPOV_PriceImp_(Bid=100bp)
0,1,4/21/2025,IBM,Buy,309300,236.2200,0.3900,5153500,38,6.0017,30.1871,104.6632,10.2630,0.5402,0.0797,0.0747,0.1906
1,2,4/21/2025,TRV,Buy,145730,249.5900,0.3600,1616000,33,9.0179,32.7824,118.4260,13.3916,0.8116,0.0823,0.0826,0.1678
2,3,4/21/2025,EMR,Buy,324790,96.4200,0.5500,3611800,56,8.9925,45.0044,180.6728,22.6610,0.8093,0.0894,0.0926,0.1044
3,4,4/21/2025,IFF,Buy,21280,72.7100,0.4300,2113100,39,1.0071,17.3898,47.2699,1.7674,0.0906,0.0659,0.0385,0.5000
4,5,4/21/2025,SO,Buy,50150,90.2300,0.2300,5008400,22,1.0013,10.8548,25.2118,0.9913,0.0901,0.0583,0.0363,0.5000
5,6,4/21/2025,MSFT,Buy,262400,359.1200,0.4000,26193600,45,1.0018,16.4414,43.8566,2.0286,0.0902,0.0649,0.0428,0.5000
6,7,4/21/2025,DG,Buy,220160,95.6100,0.4600,5504300,8,3.9998,29.6420,100.7782,1.4399,0.3600,0.0784,0.0256,0.1961
7,8,4/21/2025,GDDY,Buy,51800,165.2100,0.4000,1731600,56,2.9915,24.1118,75.7865,7.5385,0.2692,0.0738,0.0713,0.2747
8,9,4/21/2025,FAST,Buy,86430,80.2900,0.4100,4317600,41,2.0018,21.3408,63.5456,3.6933,0.1802,0.0707,0.0515,0.3422
9,10,4/21/2025,BX,Buy,384540,120.2200,0.7400,6410200,18,5.9989,48.7948,198.5442,4.8591,0.5399,0.0903,0.0380,0.0927


## 9 · Save results to CSV

In [12]:
results.to_csv('Case_Study_02_PreTrade_Results.csv', index=False)
print('Saved: Case_Study_02_PreTrade_Results.csv')
print(f'Rows: {len(results)},  Columns: {len(results.columns)}')

Saved: Case_Study_02_PreTrade_Results.csv
Rows: 25,  Columns: 17
